# Séance 3 — Qualité des données et couche Silver
*Session 3 — Data quality and the Silver layer*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/03_qualite_silver.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ● séance complète |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | ● séance complète |

**À la fin de cette séance, vous saurez :**

1. normaliser des champs hétérogènes (montants, devises, codes) ;
2. dédoublonner selon une règle explicite et traçable ;
3. écrire des contrôles qualité configurables et mettre les rejets en quarantaine ;
4. lire et interpréter un rapport de qualité (seuil bloquant).

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

Les règles de qualité ne sont **pas** codées en dur : elles vivent dans la configuration (`qualite:` dans
`conf/config_cours.yaml`). Changer un seuil ne demande donc aucune modification de code.

In [ ]:
cfg = preparer("nettoyer")
q = cfg["qualite"]
print(q)
rapport = lire_json(cfg.chemin("artefacts") / "qualite" / "rapport_qualite.json")
rapport

In [ ]:
rejets = table("quarantaine", "rejets")
rejets[["transaction_id", "montant", "devise", "motif_rejet"]].head()

In [ ]:
bronze = table("bronze", "transactions")
bronze[["montant", "devise"]].sample(8, random_state=1)

### Exercice 3.1 — 🟢 Tous les parcours

Écrivez `normaliser_montant(serie)` : retire les espaces, remplace la virgule décimale par un point et convertit en
nombre (`NaN` si illisible).

In [ ]:
# VOTRE CODE ICI
def normaliser_montant(serie):
    ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_r = normaliser_montant(pd.Series(["12,50", " 3.2 ", "abc"]))
verifier(_r.iloc[0] == 12.5 and _r.iloc[1] == 3.2 and pd.isna(_r.iloc[2]))

### Exercice 3.2 — 🟢 Tous les parcours

Dédoublonnez Bronze : une transaction (`transaction_id`, après `strip()`) = une ligne; en cas de doublon, on garde
l'ingestion **la plus récente** (`_horodatage_ingestion`). Stockez le résultat dans `dedoublonne`.

In [ ]:
# VOTRE CODE ICI
dedoublonne = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(len(dedoublonne) == rapport["lignes_apres_dedoublonnage"], "Même nombre de lignes que le pipeline Spark")
verifier(dedoublonne["transaction_id"].is_unique)

### Exercice 3.3 — 🟢 Tous les parcours

Écrivez `motif_rejet(df, q)` qui renvoie une `Series` : le **premier** motif de rejet applicable, ou `None`.
Ordre des règles : `champ_obligatoire_manquant` (une colonne de `q["colonnes_obligatoires"]` manquante), `montant_illisible`,
`montant_negatif_ou_nul` (< `montant_min`), `montant_hors_bornes` (> `montant_max`), `devise_non_autorisee` (après majuscules),
`horodatage_illisible`, `geolocalisation_invalide` (latitude manquante ou hors [-90, 90]). Indice : `np.select`.

In [ ]:
# VOTRE CODE ICI
def motif_rejet(df, q):
    ...

motifs = motif_rejet(dedoublonne, q)
motifs.value_counts()
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_obtenu = motifs.value_counts().to_dict()
verifier(_obtenu == rapport["detail_rejets"], f"Attendu {rapport['detail_rejets']}, obtenu {_obtenu}")
verifier(int(motifs.isna().sum()) == rapport["lignes_silver"], "Le nombre de lignes conformes doit égaler Silver")

**❓ Question (🔶 Gestion — décision d'affaires) —** Le pipeline échoue si plus de 5 % des lignes sont rejetées (`seuil_rejet_bloquant`). Qui devrait fixer ce seuil, et que coûte un seuil trop bas ou trop haut ?

*Votre réponse ici.*

## Bilan

- Silver = données normalisées, dédoublonnées et contrôlées; les rejets vont en quarantaine avec leur motif.
- Les règles sont dans la configuration : auditables et modifiables sans code.
- Un rapport de qualité et un seuil bloquant protègent les couches en aval.

**Pour aller plus loin :**

- `src/fraude360/quality/silver.py` (version Spark des mêmes règles).
- Great Expectations / Delta Live Tables *expectations* : les équivalents industriels.

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*